In [2]:
#Imports generales
import math
import numpy as np
import random

#Imports gráficos
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib import rc
import re

#Imports AG
import sys
!{sys.executable} -m pip install deap
import deap
from deap import base, creator, tools

#Imports AC
import cellpylib as cpl

#Imports paralelización
#import multiprocessing
#from multiprocessing import Pool
from joblib import Parallel, delayed
import time
import os

#Metricas de similitd
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import mean_squared_error



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip3 install --upgrade pip


Parámetros del AC
NUM_STATES -> Número de estados de cada célula
VEC_SIZE -> Vecinos considerados a cada lado de la célula
CA_NUM -> Número de autómatas utilizados en la función de fitness para testear la regla
CA_SIZE -> Tamaño del autómata
CA_TIMESTEPS -> Pasos de evolución de los autómatas

Parámetros del AG
IND_SIZE -> Tamaño de los individuos
POP_SIZE -> Tamaño de la poblacion inicial
CXPB -> Probabilidad de cruce
MUTPB -> Probabilidad de mutacion
NGEN -> Número de generaciones
STOP_CONDITION -> Valor de fitness exigido en la condición de parada
FITNESS_THRESHOLD -> Valor de fitness umbral de la función de fitness adaptativa
NUM_MUT -> Número de mutaciones realizadas en cada individuo

In [25]:
NUM_STATES = 2
VEC_SIZE = 1
CA_NUM = 20
CA_SIZE_1 = 30
CA_SIZE_2 = 45
CA_TIMESTEPS = 75
P = 1
MOORE = True
NEUMANN = False

if NEUMANN:
    INPUT_SIZE = 5
elif MOORE:
    INPUT_SIZE = 9
else:
    INPUT_SIZE = 2*VEC_SIZE + 1

IND_SIZE = int(pow(NUM_STATES, INPUT_SIZE))
POP_SIZE = 150
CXPB, MUTPB, NGEN = 0.9, 1, 1000


STOP_CONDITION = 0.8
FITNESS_THRESHOLD = 0.5

NUM_MUT = 15 #Número de genes mutados en cada individuo (implementación de mutaación personalizada en función principal)



'''Patrón final deseado (bandera húngara)'''
HUN = np.zeros((CA_SIZE_1, CA_SIZE_2), dtype=int)
h = CA_SIZE_1 // 3
HUN[0:h, :] = 0     
HUN[h:2*h, :] = 1   
HUN[2*h:, :] = 2

'''Patrón final deseado (bandera austriaca)'''
AUS = np.zeros((CA_SIZE_1, CA_SIZE_2), dtype=int)
h = CA_SIZE_1 // 3
AUS[0:h, :] = 0     
AUS[h:2*h, :] = 1   
AUS[2*h:, :] = 0

'''Patrón final deseado (bandera francesa)'''
FRA = np.zeros((CA_SIZE_1, CA_SIZE_2), dtype=int)
h = CA_SIZE_1 // 3
FRA[:, 0:h] = 0     
FRA[:, h:2*h] = 1   
FRA[:, 2*h:] = 2

'''Patrón final deseado (bandera japonesa)'''
JPN = np.zeros((CA_SIZE_1, CA_SIZE_2), dtype=int)
cy, cx = CA_SIZE_1 // 2, CA_SIZE_2 // 2
r = CA_SIZE_1 // 3  # El radio es aprox un tercio de la altura
y, x = np.ogrid[:CA_SIZE_1, :CA_SIZE_2]
mascara_circulo = (x - cx)**2 + (y - cy)**2 <= r**2
JPN[mascara_circulo] = 1



            

'''Creador de fitness y de individuo
#Fitness con único objetivo (weights = (1.0,) y máximo'''
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

'''Especificación de genes, individuos y poblacion
Individuos ternarios, attr_int entre 0 y 2, y en forma de lista
'''
toolbox = base.Toolbox()
toolbox.register("attr_int", random.randint, 0, NUM_STATES - 1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_int, IND_SIZE)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

'''Regla identidad por si se desea hacer el autómata probabilistico'''
def identity_rule(IND_SIZE):
    rule = [0]*IND_SIZE
    for i in range(IND_SIZE):
        if (NEUMANN):
            ternario = entero_a_base3(i, longitud_total=5)
            rule[i] = int(ternario[0])
        elif (MOORE):
            ternario = entero_a_base3(i, longitud_total=9)
            rule[i] = int(ternario[0])
        else:
            ternario = entero_a_base3(i, longitud_total=2*VEC_SIZE + 1)
            rule[i] = int(ternario[VEC_SIZE])
    return rule
        

'''Creador de regla de transición en el formato requerido por CellPyLib a partir de diccionario de reglas.'''
def create_transition_rule(individual, probability, ca_size_1, ca_size_2):
    rule_table = np.array(individual, dtype=int)
    '''id_rule_table = np.array(identity_rule(len(individual)), dtype=int)'''
    if NEUMANN:
        powers = np.array([16, 8, 4, 2, 1], dtype=int) 
    elif MOORE:
        powers = np.array([256, 128, 64, 32, 16, 8, 4, 2, 1], dtype=int)
    def my_rule(cells, r, t):
        if hasattr(cells, 'filled'):
            cells = cells.filled(0)

        neighbors_flat = cells.flatten().astype(int)

        if r[0] == 0: 
            return 0
        if r[0] == ca_size_1 - 1: 
            return 0
        if r[1] == 0 or r[1] == ca_size_2 - 1:
            return 0
        
        if NEUMANN:
            
            relevant = np.array([
                neighbors_flat[1], 
                neighbors_flat[3], 
                neighbors_flat[4], 
                neighbors_flat[5], 
                neighbors_flat[7]
            ])
            idx = np.dot(relevant, powers)
            
        elif MOORE:
            idx = np.dot(neighbors_flat, powers)

        '''if random.random() < probability: #Construcción de la regla general con probabilidad P
            return rule_table[idx]
        else: #Caso de regla identidad con probabilidad 1 - P
            return id_rule_table[idx]'''

        return rule_table[idx]
            
    return my_rule

'''Generación aleatoria de autómatas, teniendo en cuenta la frontera fija'''
def generate_CAs(ca_num, num_states, ca_size_1, ca_size_2):
    CAs = []
    for i in range(ca_num):
        '''Todo ceros'''
        ca = np.zeros((ca_size_1, ca_size_2), dtype=int)
        '''Aleatorio'''
        #ca = np.random.randint(0, NUM_STATES, size=(ca_size_1, ca_size_2))

        '''Fijamos cruz central'''
        ca[CA_SIZE_1//2, CA_SIZE_2//2] = 1
        ca[CA_SIZE_1//2 + 1, CA_SIZE_2//2] = 1
        ca[CA_SIZE_1//2 - 1, CA_SIZE_2//2] = 1
        ca[CA_SIZE_1//2, CA_SIZE_2//2 + 1] = 1
        ca[CA_SIZE_1//2, CA_SIZE_2//2 - 1] = 1
        ca = np.expand_dims(ca, axis=0)
        CAs.append(ca)
    return CAs

'''Conversión de enteros a base3, necesaria para determinar los vecindarios asociados a cada índice de las reglas'''
def entero_a_base3(numero, longitud_total=0):
    if numero == 0:
        return "0".zfill(longitud_total)

    ternario = ""
    n = numero

    while n > 0:
        residuo = n % 3
        ternario = str(residuo) + ternario 
        n = n // 3

    return ternario.zfill(longitud_total)

'''Creación del diccionario de reglas. Recibe un individuo (array de elementos ternarios), y para cada elemento
calcula su índice en ternario (vecindario) y le asigna el nuevo estado de la célula central'''
def gen_rule_dict(individual):
    rule_dict = {}
    for i in range(len(individual)):
        if (NEUMANN):
            ternario = entero_a_base3(i, longitud_total=5)
            rule_dict[ternario] = individual[i]
        elif (MOORE):
            ternario = entero_a_base3(i, longitud_total=9)
            rule_dict[ternario] = individual[i]
        else:
            ternario = entero_a_base3(i, longitud_total=2*VEC_SIZE + 1)
            rule_dict[ternario] = individual[i]

    return rule_dict

'''Evolucion de los autómatas de la lista CAs mediante mi_regla'''
def evolved_CAs(CAs, mi_regla):
    evolved_CAs = []
    for i in range(len(CAs)):
        evolved_CA = cpl.evolve2d(CAs[i], timesteps=CA_TIMESTEPS, neighbourhood = "Moore", apply_rule=mi_regla)
        evolved_CAs.append(evolved_CA)

    return evolved_CAs

'''Metrica accuracy'''
def accuracy_metric(ca_final, target):
    return (np.sum(ca_final == target)/(CA_SIZE_1*CA_SIZE_2))

'''Metrica Jaccard'''
def jaccard_metric(ca_final, target):
    jaccard_states = []
    for state in range(NUM_STATES):
        intersection = np.sum((ca_final == state) & (target == state))
        union = np.sum((target == state) | (ca_final == state))
        if union == 0:
            jaccard_states.append(1.0)  # Si no hay elementos en la unión, consideramos Jaccard como 1
        else:
            jaccard_states.append(intersection / union)
    return np.mean(jaccard_states)

'''Métrica SSIM'''
'''ESTUDIAR SI AÑADIR MAS PARAMETROS'''
def ssim_metric(ca_final, target):
    return ssim(ca_final, target, data_range = NUM_STATES - 1, win_size=5) 

'''Métrica MSE'''
'''ESTUDIAR SI AÑADIR MAS PARAMETROS'''
def mse_metric(ca_final, target):
    return mean_squared_error(ca_final, target, data_range = NUM_STATES - 1) 

'''Metrica Correlation'''
def correlation_metric(ca_final, target):
    ca_mean = np.mean(ca_final)
    target_mean = np.mean(target)
    
    covariance = np.sum((ca_final - ca_mean) * (target - target_mean))
    variance_ca = np.sum((ca_final - ca_mean)**2)
    variance_target = np.sum((target - target_mean)**2)
    
    if variance_ca == 0 or variance_target == 0:
        return 0
    else:
        return covariance / np.sqrt(variance_ca * variance_target)

'''Metrica Haussdorff'''
'''ESTUDIAR SI AÑADIR MAS PARAMETROS, UMBRALIZACION, SEGMENTACION, COMO MODELAR...'''
def haussdorf_metric(ca_final, target):
    return hausdorff_distance(ca_final, target) 



'''Funcion que genera la regla a partir de un individuo, evoluciona los autómatas con esa regla y devuelve el valor de fitness 
correspondiente.'''
def rule_and_evolve(ca_num, individual, CAs, ca_size_1, ca_size_2):
    #Diccionario de reglas
    mi_regla = create_transition_rule(individual, P, ca_size_1, ca_size_2)
    
    #Evolución
    CAs = evolved_CAs(CAs, mi_regla)

    '''Estudiar métricas para medir fitness'''
    fitness_values = []
    for ca in CAs:
        ssim = ssim_metric(ca[-1], JPN)
        jaccard = jaccard_metric(ca[-1], JPN)
        fit = 0.8 * ssim + 0.2 * jaccard
        counts = np.sum(ca[-1] == 0)
        total_cells = ca[-1].size
        if np.any(counts > (0.97 * total_cells)):
            fit = fit * 0.5  

        fitness_values.append(fit)
    
        '''metrics_accuracy.append(accuracy_metric(ca[-1], HUN))
        metrics_mse.append(mse_metric(ca[-1], HUN))
        metrics_correlation.append(correlation_metric(ca[-1], HUN))
        metrics_haussdorf.append(haussdorf_metric(ca[-1], HUN))'''

    
        
    return np.mean(fitness_values)

'''Funcion de fitness adaptativa:
Genera 20 autómatas aleatoriamente, determina la regla asociada al individuo y los evoluciona con esa regla. 
Si esa regla acierta, de media, en más del 75% de las células, se repite el proceso con 100 autómátas.
Si no, se devuelve el porcentaje por debajo de 75%. De esta forma, se evita evaluar reglas que no son tan buenas
y se gana eficiencia temporal'''

def evaluate_HUN(individual):
    
    CAs = generate_CAs(1, NUM_STATES, CA_SIZE_1, CA_SIZE_2)

    res = rule_and_evolve(1, individual, CAs, CA_SIZE_1, CA_SIZE_2)

    return (res,)

'''Funcion de mutacion personalizada: selecciona un gen al azar, lo cambia a uno distinto aleatoriamente y devuelve el individuo modificado'''
def mutChangeGen(individual):
    gen = random.randint(0, len(individual) - 1)
    
    val = individual[gen]

    choices = [i for i in range(NUM_STATES) if i != val]
    new_val = random.choice(choices)
    
    individual[gen] = new_val
    
    return individual

'''Configuración del cruce y la selección (aunque no sea necesario), selecciona la mutacion personalizada (mutChangeGen) y
registra la función de evaluacion personalizada del algoritmo '''
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", mutChangeGen)
toolbox.register("select", tools.selTournament, tournsize=3)
toolbox.register("evaluate", evaluate_HUN)

def GA():
    global NUM_MUT
    max_fitness_values = []
    '''generación aleatoria de la poblacion'''
    pop = toolbox.population(n=POP_SIZE)

    '''Evaluación de toda la población paralelamente mediante la libreria joblib y asignacion de cada fitness a su individuo'''
    print('Iniciando cálculo de fitness', flush = True)
    fitnesses = Parallel(n_jobs=-1)(delayed(toolbox.evaluate)(ind) for ind in pop)
    for ind, fit in zip(pop, fitnesses):
        ind.fitness.values = fit
    print('Cálculo de fitness terminado', flush = True)

    
    for g in range(NGEN):
        print(f'--- Iniciando Generación {g} ---', flush=True)

        '''Estrategia clásica comentada'''
        '''Selección:
        Dejamos que el 20% de los mejores individuos pasen a la siguiente generación'''
        '''elites = list(map(toolbox.clone, tools.selBest(pop, k= round(0.2*POP_SIZE))))'''
        
        '''Hacemos selección por torneo para construir el 80% restante de la próxima generacion'''
        '''offspring = toolbox.select(pop, len(pop) - round(0.2*POP_SIZE))
        offspring = list(map(toolbox.clone, offspring))'''

        '''Cruce y mutación'''
        '''for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CXPB:
                toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values

        for mutant in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(mutant)
                del mutant.fitness.values'''

        '''Evaluar paralelamente individuos con fitness inválido, es decir, indiviudos cuyo fitness no se ha calculado todavía.'''
        '''invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        fitnesses = Parallel(n_jobs=-1)(delayed(toolbox.evaluate)(ind) for ind in invalid_ind)
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit'''

        '''Nueva población: 
        Construimos la población nueva juntando el 20% de elitismo (elites) con el 80% de selección por torneo (offspring)'''
        '''pop[:] = elites + offspring'''

        '''Estrategia mu + lambda'''
        '''Clonamos la poblacion y mutamos cada individuo N veces'''
        pop.sort(key=lambda x: x.fitness.values[0], reverse=True)

        n_elites = int(round(0.1 * POP_SIZE))
        n_rest   = POP_SIZE - n_elites

        # ELITES
        elites = list(map(toolbox.clone, pop[:n_elites]))

        # RESTO
        aux = list(map(toolbox.clone, pop))
        offspring = list(map(toolbox.clone, pop))

        for mutant in offspring:
            for i in range(NUM_MUT):
                toolbox.mutate(mutant)
            del mutant.fitness.values

        '''Calculamos los nuevos fitness'''
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        fitnesses = Parallel(n_jobs=-1)(delayed(toolbox.evaluate)(ind) for ind in invalid_ind)
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit


        '''Para cada individuo y su correspondiente mutante, sleccionamos aquel con mayor fitness'''
        '''pop_aux = []
        for ind, mut in zip(pop, offspring):
            if ind.fitness.values[0] > mut.fitness.values[0]:
                pop_aux.append(ind)
            else:
                pop_aux.append(mut)
        pop = pop_aux'''
        pop_aux = aux + offspring


        pop_resto = tools.selBest(pop_aux, k = n_rest)
        pop[:] = elites + pop_resto

        
        '''Descartamos el 10% de las peores reglas y las añadimos aleatoriamente -> generacion de diversidad en la población'''
        '''pop_aux1 = tools.selBest(pop_aux, k = round(POP_SIZE*0.9))
        pop_aux2 = toolbox.population(n=POP_SIZE - round(POP_SIZE*0.9))
        fitnesses = Parallel(n_jobs=-1)(delayed(toolbox.evaluate)(ind) for ind in pop_aux2)
        for ind, fit in zip(pop_aux2, fitnesses):
            ind.fitness.values = fit
            
        pop[:] = pop_aux1 + pop_aux2'''

        


        top = tools.selBest(pop, 3)

        "Estrategia exploración vs explotación: ajuste dinámico del número de mutaciones"
        if (top[0].fitness.values[0] > 0.4 and top[0].fitness.values[0] < 0.6):
            NUM_MUT = random.randint(9,11)
        elif (top[0].fitness.values[0] > 0.6 and top[0].fitness.values[0] < 0.8):
            NUM_MUT = random.randint(5,7)
        elif (top[0].fitness.values[0] > 0.8):
            NUM_MUT = random.randint(2,4)


        print(f"--- Gen {g} Completada: Max Fitness (SSIM)= {top[0].fitness.values[0]:.2f}", flush=True)
        
        '''print(f"--- Gen {g} Completada: Max Fitness (MSE)= {mse_metric:.2f}", flush=True)
        print(f"--- Gen {g} Completada: Max Fitness (Correlation)= {top[0].fitness.values[0]:.2f}", flush=True)
        print(f"--- Gen {g} Completada: Max Fitness (Hausdorff)= {top[0].fitness.values[0]:.2f}", flush=True)
        print(f"--- Gen {g} Completada: Max Fitness (Accuracy)= {top[0].fitness.values[0]:.2f}", flush=True)'''

        max_fitness_values.append(top[0].fitness.values[0])
        print(top[0])
        if top[0].fitness.values[0] >= STOP_CONDITION: #Estudiar condición de parada
            print(f"Parado en la generación {g} con fitness {top[0].fitness.values[0]}")
            return top

    return tools.selBest(pop, 3)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "


In [24]:
'''Código de colores adaptado para Japón'''
BLANCO_FONDO    = "\033[47m" # Fondo de consola blanco (opcional, para ver mejor)
BLANCO_BRILLANTE = "\033[97m"
ROJO_BRILLANTE   = "\033[91m"
RESET            = "\033[0m"

import numpy as np

def dibujar_regla(regla):
    simbolo = "■" 
    
    # Mapeo: 0 = Blanco (Fondo), 1 = Rojo (Sol)
    simbolos_map = { 
        '0': f"{BLANCO_BRILLANTE}{simbolo}{RESET}", 
        '1': f"{ROJO_BRILLANTE}{simbolo}{RESET}"
    }

    # Detectar si es Von Neumann (32 reglas aprox) o Moore (512 reglas)
    longitud_regla = len(regla)
    
    if longitud_regla <= 32:
        tipo_vecindad = "Von Neumann (5 vecinos)"
        vecinos = 5
    else:
        tipo_vecindad = "Moore (9 vecinos)"
        vecinos = 9

    print(f"--- Catálogo de Reglas: {tipo_vecindad} ---")
    print(f"Leyenda: 0={simbolos_map['0']} (Blanco), 1={simbolos_map['1']} (Rojo)")
    print("-" * 40)

    for indice in range(longitud_regla):
        
        # 1. Convertir índice a BINARIO (base 2), rellenando ceros
        config_str = np.base_repr(indice, base=num_states).zfill(vecinos)
        
        # 2. Obtener el resultado de la regla
        resultado = regla[indice]
        res_visual = simbolos_map[str(resultado)]

        print(f"Índice {indice:03d} (Binario: {config_str}):")

        # 3. DIBUJAR SEGÚN EL TIPO DE VECINDAD
        if vecinos == 9: # MOORE (3x3)
            # Asumiendo orden estándar: 0=NW, 1=N, 2=NE, 3=W, 4=Center, 5=E, 6=SW, 7=S, 8=SE
            # Fila Superior
            linea1 = f" {simbolos_map[config_str[0]]} {simbolos_map[config_str[1]]} {simbolos_map[config_str[2]]} "
            # Fila Media (Con flecha al resultado)
            linea2 = f" {simbolos_map[config_str[3]]} {simbolos_map[config_str[4]]} {simbolos_map[config_str[5]]}  ->  {res_visual}"
            # Fila Inferior
            linea3 = f" {simbolos_map[config_str[6]]} {simbolos_map[config_str[7]]} {simbolos_map[config_str[8]]} "
            
            print(linea1)
            print(linea2)
            print(linea3)

        else: # VON NEUMANN (Cruz)
            # Usando tu mapeo anterior o el estándar: [Centro, N, E, S, W] vs [N, W, C, E, S]
            # Nota: Si usas base_repr estándar, el bit más significativo suele ser el vecino 0.
            # Ajusta estos índices según cómo construyas tu 'powers' en create_transition_rule
            
            # Asumiendo orden visual estándar: N=0, W=1, C=2, E=3, S=4 (ejemplo)
            # Adaptalo a tu lógica de 'powers'
            
            # Si usamos tu lógica anterior: 
            # (Ojo: revisa tu función create_transition_rule para ver qué potencia asignaste a cada vecino)
            C = simbolos_map[config_str[2]] 
            N = simbolos_map[config_str[0]]
            W = simbolos_map[config_str[1]]
            E = simbolos_map[config_str[3]]
            S = simbolos_map[config_str[4]]

            print(f"    {N}    ")       
            print(f"  {W} {C} {E}  ->  {res_visual}") 
            print(f"    {S}    ")       

        print("-" * 20)


'''Llama al algoritmo y hace un test con las 3 mejores reglas'''
if __name__ == "__main__":
    top3 = GA()
    CA_NUM_TEST = 250
    for individual in top3:
        regla = np.array(individual)
        dibujar_regla(regla)

        
        CAs = generate_CAs(1, NUM_STATES, CA_SIZE_1, CA_SIZE_2)

        '''rule_dict = gen_rule_dict(individual)'''
        mi_regla = create_transition_rule(individual, P, CA_SIZE_1, CA_SIZE_2)

        '''Evolucion de los autómatas de prueba con la regla (ahora sí, en paralelo)'''
        CAs = Parallel(n_jobs=-1)(delayed(cpl.evolve2d)(CAs[i], timesteps=CA_TIMESTEPS, neighbourhood = "Moore", apply_rule=mi_regla) for i in range(len(CAs)))

        '''Estudio del rendimiento de la regla'''
        metrics = []
        for ca in CAs:
            metrics.append(ssim_metric(ca[-1], JPN))
                
        porcentaje_exito = (np.sum(metrics) / CA_NUM_TEST) * 100
        print("\n" + "*"*30)
        print(f"  RENDIMIENTO DE LA REGLA:")
        print(f"  FITNESS:  {porcentaje_exito:.2f}%")
        print("*"*30 + "\n")

        
        '''Graficación de la evolucion de los autómatas'''
        colores_ternarios = ['white', 'red'] 
        cmap_personal = ListedColormap(colores_ternarios)
        
        for i in range(len(CAs)):
            plt.figure(figsize=(8, 4))
            plt.imshow(CAs[i][-1], cmap=cmap_personal, interpolation='nearest', aspect='auto')
            plt.xlabel("Celda")
            plt.ylabel("Tiempo")
            plt.title(f"Evolución del autómata CA {i}")
            plt.show()

KeyboardInterrupt: 

In [ ]:

num_automata = 50
CA_SIZE_LIST_1 = [30]
CA_SIZE_LIST_2 = [45]
CA_TIMESTEPS_LIST = [150]


for individual in reglas:
    dibujar_regla(individual)
    for j in range(len(CA_SIZE_LIST_1)):
        
        CAs = generate_CAs(num_automata, NUM_STATES, CA_SIZE_LIST_1[j], CA_SIZE_LIST_2[j])
    
        rule_dict = gen_rule_dict(individual)
        mi_regla = create_transition_rule(individual, P, CA_SIZE_LIST_1[j], CA_SIZE_LIST_2[j])

        CAs = Parallel(n_jobs=-1)(delayed(cpl.evolve2d)(CAs[i], timesteps=CA_TIMESTEPS_LIST[j], neighbourhood = "von Neumann", apply_rule=mi_regla) for i in range(len(CAs)))
    
        #Estudiamos los resultados
        common_cells = []
        for ca in CAs:
            common_cells.append(np.sum(ca[-1] == JPN)/(CA_SIZE_LIST_1[j]*CA_SIZE_LIST_2[j]))
                
        porcentaje_exito = (np.sum(common_cells) / num_automata) * 100
        print("\n" + "*"*30)
        print(f"  RENDIMIENTO DE LA REGLA:")
        print(f"  ACCURACY:  {porcentaje_exito:.2f}%")
        print("*"*30 + "\n")
    
        # 4. Definir colores y graficar
        colores_ternarios = ['red', 'white', 'green'] 
        cmap_personal = ListedColormap(colores_ternarios)
        
        for i in range(len(CAs)):
            plt.figure(figsize=(8, 4))
            plt.imshow(CAs[i][-1], cmap=cmap_personal, interpolation='nearest', aspect='auto')
            plt.xlabel("Celda")
            plt.ylabel("Tiempo")
            plt.title(f"Evolución del autómata CA {i}")
            plt.show()
    

In [ ]:
#Visualización de la evolución de un autómata aleatorio con la mejor regla encontrada

CA_SIZE_1 = 18
CA_SIZE_2 = 30
CA_TIMESTEPS = 200

colores = ['#CE2939', '#FFFFFF', '#477050']
mi_cmap = ListedColormap(colores)
regla_1 = [0, 0, 0, 0, 0, 0, 1, 2, 2, 0, 0, 1, 0, 1, 1, 0, 2, 2, 1, 2, 2, 1, 0, 0, 1, 0, 2, 0, 2, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 2, 1, 1, 1, 0, 2, 1, 0, 0, 2, 1, 0, 1, 2, 2, 2, 1, 0, 0, 2, 1, 1, 0, 1, 2, 2, 1, 2, 0, 2, 0, 0, 2, 2, 0, 1, 0, 1, 1, 2, 0, 0, 2, 2, 1, 1, 2, 0, 1, 0, 1, 0, 1, 0, 1, 0, 2, 1, 0, 1, 2, 0, 1, 2, 1, 1, 1, 1, 1, 0, 2, 2, 0, 1, 1, 1, 1, 1, 1, 2, 1, 0, 1, 2, 0, 0, 1, 2, 2, 1, 2, 0, 1, 0, 0, 1, 2, 1, 2, 1, 0, 1, 0, 1, 1, 1, 0, 2, 0, 2, 0, 2, 1, 1, 2, 1, 1, 2, 2, 2, 1, 1, 1, 0, 1, 0, 2, 1, 1, 2, 1, 0, 2, 0, 1, 1, 1, 2, 1, 0, 0, 1, 2, 1, 2, 0, 2, 1, 2, 0, 1, 2, 1, 2, 0, 0, 2, 2, 1, 0, 0, 2, 2, 1, 1, 2, 1, 0, 1, 1, 2, 2, 0, 1, 0, 2, 2, 0, 1, 0, 2, 0, 2, 1, 1, 2, 0, 1, 1, 0, 2, 2, 1, 2, 2, 1, 2, 0, 2]
regla_2 = [0, 0, 0, 0, 0, 0, 1, 2, 1, 0, 0, 2, 0, 1, 0, 0, 2, 0, 0, 2, 0, 1, 1, 2, 1, 1, 2, 0, 2, 0, 0, 1, 0, 0, 1, 2, 2, 1, 0, 1, 1, 0, 1, 1, 2, 2, 1, 1, 2, 2, 1, 0, 1, 2, 1, 0, 0, 2, 1, 2, 1, 0, 0, 2, 1, 1, 2, 1, 2, 1, 0, 2, 0, 2, 0, 0, 2, 2, 0, 1, 0, 1, 1, 2, 1, 0, 1, 2, 1, 1, 2, 0, 1, 0, 1, 0, 2, 1, 0, 0, 2, 0, 0, 0, 2, 0, 1, 2, 1, 1, 1, 0, 1, 2, 1, 0, 0, 1, 1, 1, 1, 1, 1, 2, 1, 0, 1, 1, 0, 2, 1, 2, 2, 1, 0, 0, 0, 0, 2, 1, 2, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 2, 0, 2, 0, 1, 1, 0, 0, 2, 1, 2, 2, 1, 1, 1, 1, 1, 2, 0, 2, 1, 1, 2, 0, 0, 2, 2, 1, 1, 1, 0, 1, 0, 1, 1, 2, 0, 2, 0, 2, 1, 2, 1, 1, 2, 1, 2, 0, 0, 2, 0, 1, 1, 2, 2, 2, 0, 0, 2, 1, 0, 1, 1, 2, 2, 0, 2, 0, 2, 1, 0, 2, 0, 2, 0, 0, 1, 1, 2, 0, 1, 1, 0, 0, 0, 1, 1, 2, 1, 2, 2, 2]
dibujar_regla(regla_1)

# 2. Asumiendo que 'best_ind' es tu mejor individuo salido del GA
# Creamos la función de la regla
regla_ganadora = create_transition_rule(regla_1, 1.0, CA_SIZE_1, CA_SIZE_2)


# 3. Inicializamos UN solo autómata para probar (con bordes fijos)
inicial = np.random.randint(0, NUM_STATES, size=(CA_SIZE_1, CA_SIZE_2))
inicial[0, :] = 0
inicial[-1, :] = 2
inicial = np.expand_dims(inicial, axis=0)

# 4. Evolucionamos (esto genera el historial completo que necesita la animación)
# cpl.evolve2d devuelve un array de forma (timesteps, alto, ancho)
ca = cpl.evolve2d(
    inicial, 
    timesteps=CA_TIMESTEPS, 
    neighbourhood="von Neumann", 
    apply_rule=regla_ganadora
)

# 5. VISUALIZACIÓN DIRECTA con cpl
cpl.plot2d_animate(ca, colormap=mi_cmap)



In [ ]:
log_data = """Incluir salida"""

# 1. Extraer datos usando Expresiones Regulares (Regex)
# Buscamos el patrón "Gen [NUMERO] ... Fitness = [DECIMAL]"
patron = r"Gen (\d+) Completada: Max Fitness \(SSIM\)= (\d+\.\d+)"
coincidencias = re.findall(patron, log_data)

# 2. Convertir los textos extraídos a listas numéricas
generaciones = [int(dato[0]) for dato in coincidencias]
fitness_values = [float(dato[1]) for dato in coincidencias]

# 3. Configuración de la Gráfica
plt.figure(figsize=(12, 6)) # Tamaño de la imagen
plt.plot(generaciones, fitness_values, 
         linewidth=2, 
         color='#1f77b4', # Azul estándar
         label='Mejor Individuo')

# Añadir puntos en los momentos donde cambia el fitness (escalones)
# Esto ayuda a visualizar cuándo ocurrieron las mutaciones exitosas
plt.scatter(generaciones, fitness_values, color='red', s=10, zorder=5)

# 4. Etiquetas y Estilo
plt.title('Evolución del Fitness (Algoritmo Genético)', fontsize=14)
plt.xlabel('Generación', fontsize=12)
plt.ylabel('Max Fitness', fontsize=12)
plt.grid(True, which='both', linestyle='--', alpha=0.7)
plt.legend()

# Mostrar valor final
max_val = max(fitness_values)
max_gen = generaciones[fitness_values.index(max_val)]
plt.annotate(f'Max: {max_val}', xy=(max_gen, max_val), xytext=(max_gen-50, max_val+0.01),
             arrowprops=dict(facecolor='black', shrink=0.05))

# 5. Mostrar
plt.tight_layout()
plt.show()

In [15]:
print(JPN)

[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0